In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The student passed the examination.",
    "The learner cleared the test.",
    "The cricket team won the match."
]

emb = model.encode(sentences, normalize_embeddings=True)

print("Shape:", emb.shape)
print("Data type:", emb.dtype)
print("Maximum sequence length:", model.max_seq_length)
print("First embedding:", emb[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

E:\Projects\python-ai-lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SANJAY\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (3, 384)
Data type: float32
Maximum sequence length: 256
First embedding: [ 3.62602323e-02  1.28252596e-01 -3.29180136e-02 -4.90245484e-02
 -2.19266117e-02 -5.16954809e-02  6.58809990e-02  3.11191077e-03
 -3.81449647e-02  8.78330320e-02  1.09304696e-01 -1.53408796e-02
 -1.83844119e-02  1.87727641e-02 -6.37791231e-02 -2.80931089e-02
 -8.00485387e-02 -2.02359915e-05 -8.75473674e-03  8.20368342e-03
 -2.34926362e-02  4.17108387e-02 -5.85414059e-02  4.91320491e-02
  1.30087212e-02  3.85066611e-03  1.30732032e-02 -8.14844072e-02
  5.33651300e-02 -2.73130070e-02 -2.91398112e-02 -1.74126104e-02
 -4.71155159e-02  5.86953387e-02 -4.73250635e-02 -4.35909107e-02
  3.26286368e-02  3.67113538e-02  1.40488064e-02  2.54669171e-02
 -7.96543900e-03 -3.49464901e-02 -2.17928346e-02  4.26188335e-02
  3.74318361e-02 -2.03870069e-02 -2.75634602e-03 -5.40947840e-02
  5.01329936e-02 -9.64593515e-03 -1.04829483e-01 -8.98975134e-02
 -7.94047341e-02 -2.73604542e-02 -3.77840810e-02  2.31387336e-02
  3.38964

In [4]:
from sentence_transformers import util

scores = util.cos_sim(emb, emb)

print(scores.round(decimals = 3))

tensor([[1.0000, 0.5470, 0.1680],
        [0.5470, 1.0000, 0.1320],
        [0.1680, 0.1320, 1.0000]])


In [3]:
norms = np.linalg.norm(emb, axis=1)

print(norms)

[1. 1. 1.]


In [5]:
import numpy as np

def top_k(query, matrix, k):
    matrix_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    normalized_matrix = matrix / matrix_norms

    normalized_query = query / np.linalg.norm(query)

    scores = normalized_matrix @ normalized_query
    indices = np.argsort(scores)[-k:][::-1]

    return indices, scores[indices]
    
query_text = "The learner succeeded in an exam"

query_emb = model.encode(
    query_text,
    normalize_embeddings=True
)

indices, scores = top_k(query_emb, emb, k=3)

for i, score in zip(indices, scores):
    print(f"Score: {score:.3f} | {sentences[i]}")

Score: 0.689 | The learner cleared the test.
Score: 0.687 | The student passed the examination.
Score: 0.153 | The cricket team won the match.


In [6]:
from sentence_transformers import util

reference_scores = util.cos_sim(query_emb, emb)[0].cpu().numpy()

print("Your scores:", scores)
print("Library scores for the same top results:",
      reference_scores[indices])

Your scores: [0.68886316 0.68659735 0.15284084]
Library scores for the same top results: [0.6888631  0.68659735 0.15284082]


In [7]:
test_sentences = [
    # Topic 1: Attendance
    "Section 4.2 of the attendance policy explains exam eligibility.",
    "Students must maintain minimum 75% attendance to write exams.",
    "Students must maintain minimum 65% attendance to write exams.",
    "Students are allowed to use calculators during examinations.",
    "Students are not allowed to use calculators during examinations.",

    # Topic 2: Academic results
    "CGPA is calculated using semester grades and credits.",
    "Cumulative grade point average represents overall academic performance.",

    # Topic 3: Cricket
    "India won the cricket match by scoring more runs.",
    "The bowler took three wickets in the final over.",
]

In [8]:
test_emb = model.encode(
    test_sentences,
    normalize_embeddings=True
)

print(test_emb.shape)

(9, 384)


In [12]:
query = "What does CGPA mean?"

query_emb = model.encode(
    query,
    normalize_embeddings=True
)

indices, scores = top_k(query_emb, test_emb, k=3)

for idx, score in zip(indices, scores):
    print(f"{score:.3f} | {test_sentences[idx]}")

0.755 | CGPA is calculated using semester grades and credits.
0.294 | Cumulative grade point average represents overall academic performance.
0.210 | Section 4.2 of the attendance policy explains exam eligibility.


In [14]:
negation_sentences = [
    "Students are allowed to use calculators during examinations.",
    "Students are allowed to use calculators during examinations."
]

negation_emb = model.encode(
    negation_sentences,
    normalize_embeddings=True
)

similarity = np.dot(negation_emb[0], negation_emb[1])

print("Cosine similarity:", round(float(similarity), 4))

Cosine similarity: 1.0


In [15]:
number_sentences = [
    "Students must maintain minimum 75% attendance to write exams.",
    "Students must maintain minimum 65% attendance to write exams."
]

number_emb = model.encode(
    number_sentences,
    normalize_embeddings=True
)

similarity = np.dot(number_emb[0], number_emb[1])

print("Cosine similarity:", round(float(similarity), 4))

Cosine similarity: 0.9696


In [17]:
query = "What does 4.2 say"
query_emb = model.encode(
    query,
    normalize_embeddings=True
)

indices, scores = top_k(query_emb, test_emb, k=3)

for idx, score in zip(indices, scores):
    print(f"{score:.3f} | {test_sentences[idx]}")

0.318 | Section 4.2 of the attendance policy explains exam eligibility.
0.124 | Cumulative grade point average represents overall academic performance.
0.116 | Students are allowed to use calculators during examinations.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
passage = (
    "The college has many departments and modern facilities. "
    "Students attend lectures and complete assignments. "
) * 100

full_embedding = model.encode(
    passage,
    normalize_embeddings=True
)

print("Maximum sequence length:", model.max_seq_length)
print("Full embedding shape:", full_embedding.shape)

In [ ]:
# Save embeddings to a file
np.save("emb.npy", emb)

# Load them back
loaded_emb = np.load("emb.npy")

print("Original shape:", emb.shape)
print("Loaded shape:", loaded_emb.shape)
print("Same values:", np.allclose(emb, loaded_emb))